In [107]:
import numpy as np


# Constants and Parameters
L = 0.2                  # Total thickness of the wall [m]
Nx = 20                  # Number of elements (nodes) in the wall
dx = L / Nx              # Thickness of each element [m]
T_initial = 20.0         # Initial temperature inside the wall [°C]
T_outside = 0.0          # Outside temperature [°C]
alpha = 0.001            # Thermal diffusivity of cellular concrete [m^2/s]
lambda_ = 0.5            # Thermal conductivity [W/(m·K)]
rho = 800                # Density [kg/m^3]
c_m = 840                # Specific heat capacity [J/(kg·K)]
A = 1.0                  # Cross-sectional area [m^2]
dt = 60                  # Time step [s]
time_steps = 1402        # Total time steps to reach steady state

# Initialize temperature array

T = np.full(Nx, T_initial)
T_new = np.zeros(Nx)


# Stability parameter (r)
r = lambda_ * dt / (c_m * rho * dx ** 2)

# Initialize temperature array
T = np.full(Nx, T_initial)  # All elements start at the initial temperature
T_new = np.zeros(Nx)        # Array to store updated temperatures

# Helper function for Thomas algorithm to solve tridiagonal matrix equation
def thomas_algorithm(a, b, c, d):
    n = len(d)
    
    c_ = np.zeros(n - 1)  # Initialize c_ array
    
    d_ = np.zeros(n)      # Initialize d_ array
    
    # Forward sweep
    c_[0] = c[0] / b[0]
    d_[0] = d[0] / b[0]
    
    for i in range(1, n - 1):
        temp = b[i] - a[i - 1] * c_[i - 1]
        c_[i] = c[i] / temp
        d_[i] = (d[i] - a[i - 1] * d_[i - 1]) / temp
    
    d_[n - 1] = (d[n - 1] - a[n - 2] * d_[n - 2]) / (b[n - 1] - a[n - 2] * c_[n - 2])

# Backward substitution
    x = np.zeros(n)
    x[-1] = d_[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i + 1]
    
    return x

# Construct the RHS vector d based on Crank-Nicolson method
    for i in range(1, Nx - 1):
        d[i - 1] = r * T[i - 1] + (2 - 2 * r) * T[i] + r * T[i + 1]

        # Solve for new temperature values using Thomas algorithm
    T_interior = thomas_algorithm(a, b, c, d)
    T[1:-1] = T_interior  # Update the interior temperatures

        
        #Time-stepping loop
for t in range(time_steps):
    # Construct the RHS vector `d` for the Crank-Nicolson method
    d = np.zeros(Nx - 2)
    for i in range(1, Nx - 1):
        d[i - 1] = r * T[i - 1] + (2 - 2 * r) * T[i] + r * T[i + 1]
    
    # Boundary conditions
    d[0] += r * T_outside  # Add outside temperature influence to the left boundary

    # Tridiagonal matrix coefficients for Crank-Nicolson
    a = np.full(Nx - 3, -r)
    b = np.full(Nx - 2, 2 + 2 * r)
    c = np.full(Nx - 3, -r)
    
    # Solve for new temperature values using Thomas algorithm
    T_interior = thomas_algorithm(a, b, c, d)
    T[1:-1] = T_interior  # Update the interior temperatures

    # Update boundary conditions
    T[0] = T_outside      # Left boundary at outside temperature
    T[-1] = T[-2]         # Right boundary (insulated, Neumann boundary)

    # Output control: print temperatures in selected elements
    if t % 100 == 0 or t == time_steps - 1:
        print(f'Time step {t}: '
              f'T[0]={T[0]:.2f}, T[1]={T[1]:.2f}, T[2]={T[2]:.2f}, T[3]={T[3]:.2f}, '
              f'T[16]={T[16]:.2f}, T[17]={T[17]:.2f}, T[18]={T[18]:.2f}, T[19]={T[19]:.2f}')

Time step 0: T[0]=0.00, T[1]=16.84, T[2]=19.50, T[3]=19.92, T[16]=19.92, T[17]=19.50, T[18]=16.84, T[19]=16.84
Time step 100: T[0]=0.00, T[1]=1.30, T[2]=2.58, T[3]=3.78, T[16]=4.87, T[17]=3.76, T[18]=2.55, T[19]=2.55
Time step 200: T[0]=0.00, T[1]=0.43, T[2]=0.86, T[3]=1.26, T[16]=1.62, T[17]=1.25, T[18]=0.85, T[19]=0.85
Time step 300: T[0]=0.00, T[1]=0.14, T[2]=0.28, T[3]=0.42, T[16]=0.54, T[17]=0.42, T[18]=0.28, T[19]=0.28
Time step 400: T[0]=0.00, T[1]=0.05, T[2]=0.09, T[3]=0.14, T[16]=0.18, T[17]=0.14, T[18]=0.09, T[19]=0.09
Time step 500: T[0]=0.00, T[1]=0.02, T[2]=0.03, T[3]=0.05, T[16]=0.06, T[17]=0.05, T[18]=0.03, T[19]=0.03
Time step 600: T[0]=0.00, T[1]=0.01, T[2]=0.01, T[3]=0.02, T[16]=0.02, T[17]=0.02, T[18]=0.01, T[19]=0.01
Time step 700: T[0]=0.00, T[1]=0.00, T[2]=0.00, T[3]=0.01, T[16]=0.01, T[17]=0.01, T[18]=0.00, T[19]=0.00
Time step 800: T[0]=0.00, T[1]=0.00, T[2]=0.00, T[3]=0.00, T[16]=0.00, T[17]=0.00, T[18]=0.00, T[19]=0.00
Time step 900: T[0]=0.00, T[1]=0.00, T[2]

In [106]:
import numpy as np


# Constants and Parameters
L = 0.2                  # Total thickness of the wall [m]
Nx = 20                  # Number of elements (nodes) in the wall
dx = L / Nx              # Thickness of each element [m]
T_initial = 20.0         # Initial temperature inside the wall [°C]
T_outside = 0.0          # Outside temperature [°C]
alpha = 0.001            # Thermal diffusivity of cellular concrete [m^2/s]
lambda_ = 0.5            # Thermal conductivity [W/(m·K)]
rho = 800                # Density [kg/m^3]
c_m = 840                # Specific heat capacity [J/(kg·K)]
A = 1.0                  # Cross-sectional area [m^2]
dt = 60                  # Time step [s]
time_steps = 1402        # Total time steps to reach steady state

# Initialize temperature array

T = np.full(Nx, T_initial)
T_new = np.zeros(Nx)


# Stability parameter (r)
r = lambda_ * dt / (c_m * rho * dx ** 2)

# Initialize temperature array
T = np.full(Nx, T_initial)  # All elements start at the initial temperature
T_new = np.zeros(Nx)        # Array to store updated temperatures

# Helper function for Thomas algorithm to solve tridiagonal matrix equation
def thomas_algorithm(a, b, c, d):
    n = len(d)
    
    c_ = np.zeros(n - 1)  # Initialize c_ array
    
    d_ = np.zeros(n)      # Initialize d_ array
    
    # Forward sweep
    c_[0] = c[0] / b[0]
    d_[0] = d[0] / b[0]
    
    for i in range(1, n - 1):
        temp = b[i] - a[i - 1] * c_[i - 1]
        c_[i] = c[i] / temp
        d_[i] = (d[i] - a[i - 1] * d_[i - 1]) / temp
    
    d_[n - 1] = (d[n - 1] - a[n - 2] * d_[n - 2]) / (b[n - 1] - a[n - 2] * c_[n - 2])

# Backward substitution
    x = np.zeros(n)
    x[-1] = d_[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i + 1]
    
    return x

# Construct the RHS vector d based on Crank-Nicolson method
    for i in range(1, Nx - 1):
        d[i - 1] = r * T[i - 1] + (2 - 2 * r) * T[i] + r * T[i + 1]

        # Solve for new temperature values using Thomas algorithm
    T_interior = thomas_algorithm(a, b, c, d)
    T[1:-1] = T_interior  # Update the interior temperatures

# Output control: print temperatures in selected elements
    if t % 100 == 0 or t == time_steps - 1:
        print(f'Time step {t}: '
              f'T[0]={T[0]:.2f}, T[1]={T[1]:.2f}, T[2]={T[2]:.2f}, T[3]={T[3]:.2f}, '
              f'T[16]={T[16]:.2f}, T[17]={T[17]:.2f}, T[18]={T[18]:.2f}, T[19]={T[19]:.2f}')

        
        #Time-stepping loop
for t in range(time_steps):
    # Construct the RHS vector `d` for the Crank-Nicolson method
    d = np.zeros(Nx - 2)
    for i in range(1, Nx - 1):
        d[i - 1] = r * T[i - 1] + (2 - 2 * r) * T[i] + r * T[i + 1]
    
    # Boundary conditions
    d[0] += r * T_outside  # Add outside temperature influence to the left boundary

    # Tridiagonal matrix coefficients for Crank-Nicolson
    a = np.full(Nx - 3, -r)
    b = np.full(Nx - 2, 2 + 2 * r)
    c = np.full(Nx - 3, -r)
    
    # Solve for new temperature values using Thomas algorithm
    T_interior = thomas_algorithm(a, b, c, d)
    T[1:-1] = T_interior  # Update the interior temperatures

    # Update boundary conditions
    T[0] = T_outside      # Left boundary at outside temperature
    T[-1] = T[-2]         # Right boundary (insulated, Neumann boundary)

    # Output control: print temperatures in selected elements
    if t % 100 == 0 or t == time_steps - 1:
        print(f'Time step {t}: '
              f'T[0]={T[0]:.2f}, T[1]={T[1]:.2f}, T[2]={T[2]:.2f}, T[3]={T[3]:.2f}, '
              f'T[16]={T[16]:.2f}, T[17]={T[17]:.2f}, T[18]={T[18]:.2f}, T[19]={T[19]:.2f}')

Time step 0: T[0]=0.00, T[1]=16.84, T[2]=19.50, T[3]=19.92, T[16]=19.92, T[17]=19.50, T[18]=16.84, T[19]=16.84
Time step 100: T[0]=0.00, T[1]=1.30, T[2]=2.58, T[3]=3.78, T[16]=4.87, T[17]=3.76, T[18]=2.55, T[19]=2.55
Time step 200: T[0]=0.00, T[1]=0.43, T[2]=0.86, T[3]=1.26, T[16]=1.62, T[17]=1.25, T[18]=0.85, T[19]=0.85
Time step 300: T[0]=0.00, T[1]=0.14, T[2]=0.28, T[3]=0.42, T[16]=0.54, T[17]=0.42, T[18]=0.28, T[19]=0.28
Time step 400: T[0]=0.00, T[1]=0.05, T[2]=0.09, T[3]=0.14, T[16]=0.18, T[17]=0.14, T[18]=0.09, T[19]=0.09
Time step 500: T[0]=0.00, T[1]=0.02, T[2]=0.03, T[3]=0.05, T[16]=0.06, T[17]=0.05, T[18]=0.03, T[19]=0.03
Time step 600: T[0]=0.00, T[1]=0.01, T[2]=0.01, T[3]=0.02, T[16]=0.02, T[17]=0.02, T[18]=0.01, T[19]=0.01
Time step 700: T[0]=0.00, T[1]=0.00, T[2]=0.00, T[3]=0.01, T[16]=0.01, T[17]=0.01, T[18]=0.00, T[19]=0.00
Time step 800: T[0]=0.00, T[1]=0.00, T[2]=0.00, T[3]=0.00, T[16]=0.00, T[17]=0.00, T[18]=0.00, T[19]=0.00
Time step 900: T[0]=0.00, T[1]=0.00, T[2]

In [5]:
# Initialize temperature array
T = np.full(Nx, T_initial)
T_new = np.zeros(Nx)



NameError: name 'Nx' is not defined

In [166]:
# Constants and Parameters
L = 0.2                  # Total thickness of the wall [m]
Nx = 20                  # Number of elements (nodes) in the wall
dx = L / Nx              # Thickness of each element [m]
T_initial = 20.0         # Initial temperature inside the wall [°C]
T_outside = 0.0          # Outside temperature [°C]
alpha = 0.001            # Thermal diffusivity of cellular concrete [m^2/s]
lambda_ = 0.5            # Thermal conductivity [W/(m·K)]
rho = 800                # Density [kg/m^3]
c_m = 840                # Specific heat capacity [J/(kg·K)]
A = 1.0                  # Cross-sectional area [m^2]
dt = 60                  # Time step [s]
time_steps = 1402        # Total time steps to reach steady state

In [167]:
# Stability parameter (r)
r = lambda_ * dt / (c_m * rho * dx ** 2)

In [168]:
# Initialize temperature array
T = np.full(Nx, T_initial)  # All elements start at the initial temperature
T_new = np.zeros(Nx)        # Array to store updated temperatures

In [174]:
# Coefficients for the tridiagonal matrix system
a = np.full(Nx - 2, -r)              # Sub-diagonal
b = np.full(Nx - 2, 2 + 2 * r)       # Main diagonal
c = np.full(Nx - 2, -r)              # Super-diagonal
d = np.zeros(Nx - 2)                 # Right-hand side


In [175]:
# Helper function for Thomas algorithm to solve tridiagonal matrix equation
def thomas_algorithm(a, b, c, d):
    n = len(d)
    c_ = np.zeros(n - 1)  # Initialize c_ array
    d_ = np.zeros(n)      # Initialize d_ array

In [177]:
 # Forward sweep
c_[0] = c[0] / b[0]
d_[0] = d[0] / b[0]
for i in range(1, n - 1):
        temp = b[i] - a[i - 1] * c_[i - 1]
        c_[i] = c[i] / temp
        d_[i] = (d[i] - a[i - 1] * d_[i - 1]) / temp
    
d_[n - 1] = (d[n - 1] - a[n - 2] * d_[n - 2]) / (b[n - 1] - a[n - 2] * c_[n - 2])


NameError: name 'c_' is not defined

In [162]:
 # Backward substitution
x = np.zeros(n)
x[-1] = d_[-1]
for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i + 1]
    
return x

NameError: name 'n' is not defined

In [138]:
# Main loop for time-stepping
for t in range(time_steps):
    # Apply boundary conditions
    T[0] = T_initial
    T[-1] = T_outside

In [140]:
# Coefficients for the tridiagonal matrix system
a = np.full(Nx - 2, -r)              # Sub-diagonal
b = np.full(Nx - 2, 2 + 2 * r)       # Main diagonal
c = np.full(Nx - 2, -r)              # Super-diagonal
d = np.zeros(Nx - 2)                 # Right-hand side


In [142]:
# Construct the RHS vector d based on Crank-Nicolson method
for i in range(1, Nx - 1):
        d[i - 1] = r * T[i - 1] + (2 - 2 * r) * T[i] + r * T[i + 1]


In [146]:
# Solve for new temperature values using Thomas algorithm
T_interior = thomas_algorithm(a, b, c, d)
T[1:-1] = T_interior  # Update the interior temperatures


In [148]:
# Output control: print temperatures in selected elements
if t % 100 == 0 or t == time_steps - 1:
        print(f'Time step {t}: '
              f'T[0]={T[0]:.2f}, T[1]={T[1]:.2f}, T[2]={T[2]:.2f}, T[3]={T[3]:.2f}, '
              f'T[16]={T[16]:.2f}, T[17]={T[17]:.2f}, T[18]={T[18]:.2f}, T[19]={T[19]:.2f}')


Time step 1401: T[0]=20.00, T[1]=nan, T[2]=nan, T[3]=nan, T[16]=nan, T[17]=nan, T[18]=nan, T[19]=0.00


In [160]:
# Note: Steady state is expected at around time step 1402.
